In [2]:
import pandas as pd
import numpy as np

In [10]:
def run_pipeline():
    print("Fetching raw data...")
    # Load raw dataset from source
    url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
    df = pd.read_csv(url)

print("Cleaning & processing data...")


Cleaning & processing data...


In [13]:
# 1. Safely clean TotalCharges handling both numbers and blank strings
df['TotalCharges'] = pd.to_numeric(
    df['TotalCharges'].astype(str).str.strip(), 
    errors='coerce'
).fillna(0)

# 2. Map Churn column to binary integers
df['ChurnBinary'] = df['Churn'].map({'Yes': 1, 'No': 0}).fillna(0).astype(int)

In [16]:
# Feature Engineering: Calculate Risk Score
def calculate_risk(row):
    score = 0
    
    # Contract risk
    if str(row.get('Contract', '')).strip() == 'Month-to-month': 
        score += 40
        
    # Internet Service risk
    if str(row.get('InternetService', '')).strip() == 'Fiber optic': 
        score += 25
        
    # Payment Method risk
    if str(row.get('PaymentMethod', '')).strip() == 'Electronic check': 
        score += 15
        
    # Safe tenure conversion to float/int
    try:
        tenure_val = float(row.get('tenure', 0))
    except (ValueError, TypeError):
        tenure_val = 0
        
    if tenure_val < 12: 
        score += 20
        
    return score
    
df['RiskScore'] = df.apply(calculate_risk, axis=1)

In [20]:
# Robust Risk Group Categorization using numpy.select (Avoids pd.cut issues)
conditions = [
    (df['RiskScore'] <= 35),
    (df['RiskScore'] > 35) & (df['RiskScore'] <= 65),
    (df['RiskScore'] > 65)
]
choices = ['Low Risk', 'Medium Risk', 'High Risk']
choices = ['Low Risk', 'Medium Risk', 'High Risk']

In [21]:
# Cast directly to string to ensure clean CSV export
df['RiskCategory'] = np.select(conditions, choices, default='Low Risk').astype(str)

In [22]:
# Feature Engineering: Customer Lifetime Value (CLV)
df['CLV'] = df['MonthlyCharges'] * df['tenure']

In [23]:
# Export output CSV
output_path = "churn_bi_dataset.csv"
df.to_csv(output_path, index=False)
print(f"Pipeline finished successfully! Output saved to '{output_path}'.")

Pipeline finished successfully! Output saved to 'churn_bi_dataset.csv'.


In [24]:
if __name__ == "__main__":
    run_pipeline()

Fetching raw data...
